In [47]:
%pip install confluent-kafka

In [49]:
from confluent_kafka import Producer
import json
import random
import time
from datetime import datetime, timezone

config = {
    "bootstrap.servers": "",
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": "",
    "sasl.password": ""
}

producer = Producer(config)

# This MUST be executed to update the active notebook variable in memory
TOPIC = "theromstat_readings"

print(f"Connected! Target topic set to: {TOPIC}")

Connected! Target topic set to: theromstat_readings


In [50]:
def delivery_report(err, msg):
    if err is not None:
        print(f"Delivery failed: {err}")
    else:
        print(
            f"Delivered to {msg.topic()} | "
            f"Partition: {msg.partition()} | "
            f"Offset: {msg.offset()}"
        )


for i in range(10):

    machine = random.choice(machines)

    event = generate_sensor_reading(machine)

    # Convert the Python dictionary to JSON bytes
    message = json.dumps(event).encode("utf-8")

    # Publish the event to Kafka
    producer.produce(
        topic=TOPIC,
        key=event["machine_id"].encode("utf-8"),
        value=message,
        callback=delivery_report
    )

    # Allow delivery callbacks to run
    producer.poll(0)

    print(
        f"Published event {i + 1}: "
        f"{event['machine_id']} | "
        f"Temperature: {event['temperature']}°C"
    )

    time.sleep(1)

# Wait for all outstanding messages to be delivered
remaining = producer.flush(timeout=30)

if remaining == 0:
    print("\nAll queued messages have been delivered.")
else:
    print(f"\nWarning: {remaining} messages remain undelivered.")

Published event 1: MACHINE_002 | Temperature: 58.72°C
Delivered to theromstat_readings | Partition: 0 | Offset: 134
Published event 2: MACHINE_003 | Temperature: 55.92°C
Delivered to theromstat_readings | Partition: 0 | Offset: 135
Published event 3: MACHINE_001 | Temperature: 52.53°C
Delivered to theromstat_readings | Partition: 2 | Offset: 248
Published event 4: MACHINE_003 | Temperature: 47.58°C
Delivered to theromstat_readings | Partition: 0 | Offset: 136
Published event 5: MACHINE_002 | Temperature: 46.66°C
Delivered to theromstat_readings | Partition: 0 | Offset: 137
Published event 6: MACHINE_001 | Temperature: 70.68°C
Delivered to theromstat_readings | Partition: 2 | Offset: 249
Published event 7: MACHINE_002 | Temperature: 57.48°C
Delivered to theromstat_readings | Partition: 0 | Offset: 138
Published event 8: MACHINE_002 | Temperature: 48.2°C
Delivered to theromstat_readings | Partition: 0 | Offset: 139
Published event 9: MACHINE_003 | Temperature: 67.4°C
Delivered to theroms

In [51]:
from confluent_kafka import Consumer

consumer_config = {
    "bootstrap.servers": "",
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": "",
    "sasl.password": "",
    "group.id": "smartsense-alert-consumer-v2",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False
}

consumer = Consumer(consumer_config)

# Corrected the topic spelling here to match your cluster's topic 'theromstat_readings'
consumer.subscribe(["theromstat_readings"])

print("SmartSense alert consumer initialized.")

SmartSense alert consumer initialized.


In [52]:
def detect_anomaly(event):
    alerts = []

    if event["temperature"] > 90:
        alerts.append({
            "alert_type": "HIGH_TEMPERATURE",
            "severity": "CRITICAL",
            "message": (
                f"Machine temperature is "
                f"{event['temperature']}°C"
            )
        })

    if event["vibration"] > 7:
        alerts.append({
            "alert_type": "HIGH_VIBRATION",
            "severity": "WARNING",
            "message": (
                f"Machine vibration is "
                f"{event['vibration']} mm/s"
            )
        })

    if event["pressure"] < 90 or event["pressure"] > 120:
        alerts.append({
            "alert_type": "ABNORMAL_PRESSURE",
            "severity": "WARNING",
            "message": (
                f"Machine pressure is "
                f"{event['pressure']} kPa"
            )
        })

    return alerts

In [53]:
from confluent_kafka import Producer
import json
from datetime import datetime, timezone

# Updated to use the correct 'config' dictionary defined in cell mVsxfwCuLIH4
alert_producer = Producer(config)

ALERT_TOPIC = "machine_alerts"


def publish_alert(alert):
    alert_producer.produce(
        topic=ALERT_TOPIC,
        key=alert["machine_id"].encode("utf-8"),
        value=json.dumps(alert).encode("utf-8")
    )

    # Wait for the alert to be delivered.
    remaining = alert_producer.flush(timeout=10)

    if remaining:
        raise RuntimeError(
            f"{remaining} alert message(s) remain undelivered"
        )


In [ ]:
import json
from datetime import datetime, timezone

print("SmartSense is monitoring machine readings.")
print("Run this cell and interrupt it when you want to stop.")

try:
    while True:
        msg = consumer.poll(1.0)

        if msg is None:
            continue

        if msg.error():
            print("Consumer error:", msg.error())
            continue

        try:
            event = json.loads(msg.value().decode("utf-8"))

            # Validate the event structure.
            required_fields = {
                "event_id",
                "machine_id",
                "temperature",
                "vibration",
                "pressure"
            }

            if not required_fields.issubset(event):
                print("Skipping event with missing fields:", event)
                consumer.commit(message=msg, asynchronous=False)
                continue

            alerts = detect_anomaly(event)

            if alerts:
                for detected_alert in alerts:
                    alert = {
                        "alert_id": (
                            f"{event['event_id']}_"
                            f"{detected_alert['alert_type']}"
                        ),
                        "machine_id": event["machine_id"],
                        "event_id": event["event_id"],
                        "temperature": event["temperature"],
                        "vibration": event["vibration"],
                        "pressure": event["pressure"],
                        "alert_type": detected_alert["alert_type"],
                        "severity": detected_alert["severity"],
                        "message": detected_alert["message"],
                        "alert_timestamp": (
                            datetime.now(timezone.utc).isoformat()
                        )
                    }

                    publish_alert(alert)

                    print(
                        f"ALERT: {alert['machine_id']} | "
                        f"{alert['severity']} | "
                        f"{alert['message']}"
                    )

            # Commit only after processing and publishing alerts.
            consumer.commit(
                message=msg,
                asynchronous=False
            )

        except (json.JSONDecodeError, KeyError, TypeError) as e:
            print("Invalid event:", e)

except KeyboardInterrupt:
    print("\nStopping SmartSense consumer.")

finally:
    consumer.close()

SmartSense is monitoring machine readings.
Run this cell and interrupt it when you want to stop.
ALERT: MACHINE_002 | CRITICAL | Machine temperature is 104.47°C
